## TinyML project - Training Pipeline

This notebook implements the full training pipeline for a TinyML-based speech command recognition system.

The goal is to train a lightweight 2D CNN model on MFCC features extracted from the Google Speech Commands dataset, ensuring that the preprocessing pipeline exactly matches the one implemented on the ESP32.

Key steps include:
- Loading and filtering a subset of 10 speech commands  
- Extracting MFCC features using a custom pipeline aligned with ESP32 implementation  
- Training and evaluating a 2D CNN model using TensorFlow/Keras  
- Converting the trained model into a TensorFlow Lite (float) format  

The resulting `.tflite` model is designed to be deployed on an ESP32 using TensorFlow Lite Micro for real-time, on-device speech recognition.


Install required libraries for audio processing, dataset loading, and model evaluation


In [ ]:
!pip install librosa numpy==1.26.4 datasets==2.18.0 scikit-learn

## Library Imports

The following libraries are used to implement the TinyML training pipeline:

- Signal processing: NumPy, SciPy (DCT)  
- Audio preprocessing: Librosa  
- Dataset handling: Hugging Face Datasets  
- Model training: TensorFlow / Keras  
- Evaluation: Scikit-learn  

In [ ]:
import numpy as np
from scipy.fftpack import dct
import tensorflow as tf
import librosa
from datasets import load_dataset
from sklearn.metrics import classification_report

## Configuration Parameters

This section defines the key parameters used for audio processing and model training.

Audio / MFCC Parameters
- **Sample Rate:** 16000 Hz (standard for speech recognition)  
- **Frame Size:** 512 samples per frame  
- **MFCC Coefficients:** 13 features per frame  
- **Mel Filters:** 26 filter banks  
- **Max Frames:** 100 (fixed-length input for the model)  

Training Parameters
- **Batch Size:** 32  
- **Epochs:** 50  

In [ ]:
SAMPLE_RATE = 16000
FRAME_SIZE = 512
MFCC_COUNT = 13
MEL_FILTERS = 26
MAX_FRAMES = 100

BATCH_SIZE = 32
EPOCHS = 50

## Dataset Loading and Exploration

The Speech Commands dataset is loaded using the Hugging Face `datasets` library. The dataset is inspected to understand its structure, labels, and audio format.

A subset of 10 commands is selected for classification. Additionally, a sample audio waveform is played to validate data integrity and label correctness before preprocessing.

In [ ]:
dataset = load_dataset("google/speech_commands", "v0.02", trust_remote_code=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/84848 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/9982 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4890 [00:00<?, ? examples/s]

In [ ]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['file', 'audio', 'label', 'is_unknown', 'speaker_id', 'utterance_id'],
        num_rows: 84848
    })
    validation: Dataset({
        features: ['file', 'audio', 'label', 'is_unknown', 'speaker_id', 'utterance_id'],
        num_rows: 9982
    })
    test: Dataset({
        features: ['file', 'audio', 'label', 'is_unknown', 'speaker_id', 'utterance_id'],
        num_rows: 4890
    })
})


In [ ]:
print(dataset["train"][0])

{'file': 'backward/2356b88d_nohash_0.wav', 'audio': {'path': 'backward/2356b88d_nohash_0.wav', 'array': array([ 0.        ,  0.        ,  0.        , ..., -0.00012207,
       -0.00015259, -0.00012207]), 'sampling_rate': 16000}, 'label': 30, 'is_unknown': True, 'speaker_id': '2356b88d', 'utterance_id': 0}


In [ ]:
labels = dataset["train"].features["label"].names

print("Number of classes:", len(labels))
print("Classes:", labels)

Number of classes: 36
Classes: ['yes', 'no', 'up', 'down', 'left', 'right', 'on', 'off', 'stop', 'go', 'zero', 'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine', 'bed', 'bird', 'cat', 'dog', 'happy', 'house', 'marvin', 'sheila', 'tree', 'wow', 'backward', 'forward', 'follow', 'learn', 'visual', '_silence_']


In [ ]:
COMMANDS = ["yes","no","on","off","up","down","left","right","stop","go"]

In [ ]:
import IPython.display as ipd

# Find the index of a sample with label "down"
for i, sample in enumerate(dataset["train"]):
    if labels[sample["label"]] == "down":
        play_sample = sample
        break  # stop at the first example

# Extract audio array and sample rate
audio_array = play_sample["audio"]["array"]
sample_rate = play_sample["audio"]["sampling_rate"]

# Play the audio
ipd.display(ipd.Audio(audio_array, rate=sample_rate))

# Print the label
print("Label:", labels[play_sample["label"]])

Label: down


## Dataset Filtering and Label Encoding

A label-to-index mapping is created to convert command names into numerical labels. The dataset is then filtered to retain only the selected subset of commands across all splits.

This ensures consistency between the dataset and the model output layer, defined by `NUM_CLASSES`.

In [ ]:
label_to_index = {cmd:i for i,cmd in enumerate(COMMANDS)}

def filter_commands(example):
    return labels[example["label"]] in COMMANDS

dataset["train"] = dataset["train"].filter(filter_commands)
dataset["validation"] = dataset["validation"].filter(filter_commands)
dataset["test"] = dataset["test"].filter(filter_commands)

NUM_CLASSES = len(COMMANDS)

print("Classes:", COMMANDS)
print("Number of Classes:", NUM_CLASSES)

Filter:   0%|          | 0/84848 [00:00<?, ? examples/s]

Filter:   0%|          | 0/9982 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4890 [00:00<?, ? examples/s]

Classes: ['yes', 'no', 'on', 'off', 'up', 'down', 'left', 'right', 'stop', 'go']
Number of Classes: 10


## MFCC Feature Extraction and Preprocessing

This section implements the complete audio preprocessing pipeline used for training and deployment.

The main goal is to ensure that MFCC feature extraction in Python matches the ESP32 implementation as closely as possible. This is critical so that the model sees the same type of input during training and during on-device inference.

### What this section does

- Defines audio and MFCC constants that match the ESP32 configuration  
- Implements helper functions to convert between Hertz and Mel scale  
- Builds a custom Mel filterbank  
- Builds a custom DCT matrix for MFCC computation  
- Extracts MFCC features from a single audio frame  
- Applies streaming MFCC extraction across the full audio signal  
- Resamples and normalizes audio when needed  
- Pads or truncates MFCC sequences to a fixed size  
- Adds the channel dimension required by the CNN input  
- Converts command labels into integer class indices  

### Why this matters

Matching the preprocessing pipeline between Python and ESP32 helps reduce training-inference mismatch. This improves the reliability of the deployed TensorFlow Lite model when running real-time speech recognition on the microcontroller.


In [ ]:
from scipy.fftpack import dct  # DCT is used in MFCC computation

# =========================
# CONSTANTS TO MATCH ESP32
# =========================
SAMPLE_RATE = 16000   # Audio sample rate in Hz
FRAME_SIZE = 512      # Number of samples per analysis frame
HOP_LENGTH = 160      # Step size between consecutive frames
MFCC_COUNT = 13       # Number of MFCC coefficients to keep
MEL_FILTERS = 26      # Number of Mel filterbank channels
MAX_FRAMES = 100      # Fixed number of time frames for model input

# =========================
# MEL SCALE CONVERSIONS
# =========================
def hz_to_mel(hz):
    return 2595 * np.log10(1 + hz / 700)  # Convert frequency from Hz to Mel scale

def mel_to_hz(mel):
    return 700 * (10**(mel / 2595) - 1)  # Convert frequency from Mel scale back to Hz

# =========================
# MEL FILTERBANK CREATION
# =========================
def create_mel_filterbank():
    mel_min = hz_to_mel(0)                 # Lower frequency bound in Mel
    mel_max = hz_to_mel(SAMPLE_RATE / 2)   # Nyquist frequency in Mel

    mel_points = np.linspace(mel_min, mel_max, MEL_FILTERS + 2)  # Evenly spaced Mel points
    hz_points = mel_to_hz(mel_points)                            # Convert Mel points to Hz

    bins = np.floor((FRAME_SIZE + 1) * hz_points / SAMPLE_RATE).astype(int)  # Map Hz to FFT bins

    filterbank = np.zeros((MEL_FILTERS, FRAME_SIZE // 2))  # Initialize triangular Mel filters

    for m in range(1, MEL_FILTERS + 1):
        f_m_minus, f_m, f_m_plus = bins[m-1], bins[m], bins[m+1]  # Left, center, and right bin

        for k in range(f_m_minus, f_m):
            if k < FRAME_SIZE // 2:
                filterbank[m-1, k] = (k - f_m_minus) / (f_m - f_m_minus)  # Rising slope

        for k in range(f_m, f_m_plus):
            if k < FRAME_SIZE // 2:
                filterbank[m-1, k] = (f_m_plus - k) / (f_m_plus - f_m)  # Falling slope

    return filterbank  # Return Mel filterbank matrix

mel_filterbank = create_mel_filterbank()  # Precompute Mel filterbank once

# =========================
# DCT MATRIX CREATION
# =========================
def create_dct_matrix():
    dct_matrix = np.zeros((MFCC_COUNT, MEL_FILTERS))  # Matrix to project log-Mel energies into MFCCs

    for i in range(MFCC_COUNT):
        for j in range(MEL_FILTERS):
            scale = np.sqrt(1.0 / MEL_FILTERS) if i == 0 else np.sqrt(2.0 / MEL_FILTERS)  # Orthonormal scaling
            dct_matrix[i, j] = scale * np.cos(i * (j + 0.5) * np.pi / MEL_FILTERS)         # DCT-II basis

    return dct_matrix  # Return DCT transform matrix

dct_matrix = create_dct_matrix()  # Precompute DCT matrix once

# =========================
# SINGLE-FRAME MFCC
# =========================
def compute_mfcc_esp32(frame):

    # Apply Hann window to reduce spectral leakage
    window = np.hanning(FRAME_SIZE)
    frame = frame * window

    # Compute real FFT
    fft = np.fft.rfft(frame)

    # Compute power spectrum
    power = (np.abs(fft) ** 2) / FRAME_SIZE

    # Keep only the first 256 bins to match the ESP32 implementation
    power = power[:FRAME_SIZE // 2]

    # Project FFT power into Mel bands
    mel_energy = np.dot(mel_filterbank, power)

    # Apply log compression for dynamic range reduction
    mel_energy = np.log10(mel_energy + 1e-6)

    # Apply DCT to obtain MFCC coefficients
    mfcc = np.dot(dct_matrix, mel_energy)

    return mfcc  # Return one MFCC vector for the frame


# =========================
# STREAMING MFCC EXTRACTION
# =========================
def compute_mfcc_sequence(audio):

    frames = []  # Will store one MFCC vector per frame

    for i in range(0, len(audio) - FRAME_SIZE, HOP_LENGTH):
        frame = audio[i:i + FRAME_SIZE]  # Extract one frame from the audio stream

        if len(frame) < FRAME_SIZE:
            continue  # Skip incomplete frames

        mfcc = compute_mfcc_esp32(frame)  # Compute MFCCs for the current frame
        frames.append(mfcc)               # Append to the sequence

    return np.array(frames)  # Return full MFCC time sequence


# =========================
# DATASET PREPROCESSING
# =========================
def preprocess(example):

    audio = example["audio"]["array"]                  # Raw waveform
    sr = example["audio"]["sampling_rate"]             # Original sample rate

    # Resample audio if needed to match the ESP32 pipeline
    if sr != SAMPLE_RATE:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=SAMPLE_RATE)

    # Normalize waveform amplitude to match ESP32 scaling
    audio = audio / (np.max(np.abs(audio)) + 1e-6)

    # Extract streaming MFCC features
    mfcc = compute_mfcc_sequence(audio)

    # Pad with zeros or truncate to ensure fixed-length input
    if mfcc.shape[0] < MAX_FRAMES:
        pad = MAX_FRAMES - mfcc.shape[0]
        mfcc = np.pad(mfcc, ((0, pad), (0, 0)))
    else:
        mfcc = mfcc[:MAX_FRAMES]

    # Add channel dimension so shape becomes (100, 13, 1)
    mfcc = np.expand_dims(mfcc, -1)

    label_name = labels[example["label"]]    # Convert dataset label index to command name
    label = label_to_index[label_name]       # Convert command name to model class index

    return {
        "audio": mfcc.astype(np.float32),    # Model input features
        "label": label                       # Integer class label
    }

Dataset pipeline

## Dataset Preparation

The dataset is preprocessed using the custom MFCC pipeline and converted into TensorFlow datasets.

- Applies feature extraction to all splits  
- Converts data into `(audio, label)` format  
- Uses batching, shuffling (train only), and prefetching for performance  

The resulting datasets are ready for model training and evaluation.

In [ ]:
train_ds_hf = dataset["train"].map(preprocess, num_proc=4)
val_ds_hf = dataset["validation"].map(preprocess, num_proc=4)
test_ds_hf = dataset["test"].map(preprocess, num_proc=4)

train_ds = train_ds_hf.to_tf_dataset(
    columns=["audio"],
    label_cols=["label"],
    batch_size=BATCH_SIZE,
    shuffle=True
).prefetch(tf.data.AUTOTUNE)

val_ds = val_ds_hf.to_tf_dataset(
    columns=["audio"],
    label_cols=["label"],
    batch_size=BATCH_SIZE
).prefetch(tf.data.AUTOTUNE)

test_ds = test_ds_hf.to_tf_dataset(
    columns=["audio"],
    label_cols=["label"],
    batch_size=BATCH_SIZE
)

Map (num_proc=4):   0%|          | 0/30769 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/3703 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/4074 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/datasets/arrow_dataset.py:401: FutureWarning: The output of `to_tf_dataset` will change when a passing single element list for `labels` or `columns` in the next datasets version. To return a tuple structure rather than dict, pass a single string.
Old behaviour: columns=['a'], labels=['labels'] -> (tf.Tensor, tf.Tensor)  
             : columns='a', labels='labels' -> (tf.Tensor, tf.Tensor)  
New behaviour: columns=['a'],labels=['labels'] -> ({'a': tf.Tensor}, {'labels': tf.Tensor})  
             : columns='a', labels='labels' -> (tf.Tensor, tf.Tensor) 
  warnings.warn(


## Model Architecture

This section defines and compiles a lightweight 2D Convolutional Neural Network (CNN) for speech command classification.

### Input

- Shape: **(100, 13, 1)**  
  - 100 time frames  
  - 13 MFCC coefficients  
  - 1 channel  

### Architecture

- **Conv2D (8 filters, 3×3, ReLU)**  
- **MaxPooling (2×2)**  
- **Conv2D (16 filters, 3×3, ReLU)**  
- **MaxPooling (2×2)**  
- **Flatten**  
- **Dense (32 units, ReLU)**  
- **Output Layer (Softmax)** → 10 classes  

### Compilation

- **Optimizer:** Adam  
- **Loss Function:** Sparse Categorical Crossentropy  
- **Metric:** Accuracy  

### Summary

The model is designed to be lightweight and efficient, making it suitable for deployment on resource-constrained devices such as the ESP32.


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(MAX_FRAMES, MFCC_COUNT, 1)),
    tf.keras.layers.Conv2D(8, (3,3), activation="relu"),
    tf.keras.layers.MaxPool2D((2,2)),
    tf.keras.layers.Conv2D(16, (3,3), activation="relu"),
    tf.keras.layers.MaxPool2D((2,2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(NUM_CLASSES, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 98, 11, 8)      │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 49, 5, 8)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 47, 3, 16)      │         1,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 23, 1, 16)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 368)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │        11,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           330 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,386 (52.29 KB)

 Trainable params: 13,386 (52.29 KB)

 Non-trainable params: 0 (0.00 B)

## Model Training

The model is trained using the training dataset with validation monitoring.

Early stopping is applied to prevent overfitting by stopping training when validation loss no longer improves and restoring the best weights.

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stopping]
)

Epoch 1/50
962/962 ━━━━━━━━━━━━━━━━━━━━ 267s 276ms/step - accuracy: 0.4961 - loss: 1.4166 - val_accuracy: 0.6190 - val_loss: 1.0887
Epoch 2/50
962/962 ━━━━━━━━━━━━━━━━━━━━ 265s 275ms/step - accuracy: 0.6909 - loss: 0.8891 - val_accuracy: 0.7189 - val_loss: 0.8098
Epoch 3/50
962/962 ━━━━━━━━━━━━━━━━━━━━ 276s 287ms/step - accuracy: 0.7469 - loss: 0.7322 - val_accuracy: 0.7580 - val_loss: 0.6758
Epoch 4/50
962/962 ━━━━━━━━━━━━━━━━━━━━ 268s 279ms/step - accuracy: 0.7788 - loss: 0.6399 - val_accuracy: 0.7804 - val_loss: 0.6388
Epoch 5/50
962/962 ━━━━━━━━━━━━━━━━━━━━ 262s 272ms/step - accuracy: 0.7979 - loss: 0.5835 - val_accuracy: 0.7605 - val_loss: 0.6526
Epoch 6/50
962/962 ━━━━━━━━━━━━━━━━━━━━ 267s 277ms/step - accuracy: 0.8116 - loss: 0.5427 - val_accuracy: 0.8002 - val_loss: 0.5738
Epoch 7/50
962/962 ━━━━━━━━━━━━━━━━━━━━ 262s 273ms/step - accuracy: 0.8249 - loss: 0.5082 - val_accuracy: 0.8164 - val_loss: 0.5339
Epoch 8/50
962/962 ━━━━━━━━━━━━━━━━━━━━ 265s 275ms/step - accuracy: 0.8347 -

## Model Evaluation

This section evaluates the trained model using the test dataset.

### Test Accuracy

- The model is evaluated on unseen data (`test_ds`)  
- Overall performance is measured using **accuracy**  

### Classification Report

A detailed evaluation is generated using Scikit-learn:

- **Precision** → Correct positive predictions  
- **Recall** → Ability to find all relevant samples  
- **F1-score** → Balance between precision and recall  

### Result

The classification report provides per-class performance for all commands, offering deeper insight into how well the model distinguishes between different speech commands.

In [ ]:
results = model.evaluate(test_ds)
print("Test accuracy:", results[1])

128/128 ━━━━━━━━━━━━━━━━━━━━ 32s 252ms/step - accuracy: 0.8235 - loss: 0.5809
Test accuracy: 0.823514997959137


In [ ]:
predictions = []
true_labels = []

for X,y in test_ds:
    pred = model.predict(X)
    predictions.extend(np.argmax(pred,axis=1))
    true_labels.extend(y.numpy())

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 460ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
1

In [ ]:
print(classification_report(true_labels,predictions,target_names=COMMANDS))

              precision    recall  f1-score   support

         yes       0.91      0.92      0.92       419
          no       0.75      0.80      0.77       405
          on       0.89      0.84      0.86       396
         off       0.79      0.79      0.79       402
          up       0.81      0.74      0.78       425
        down       0.77      0.80      0.78       406
        left       0.78      0.85      0.81       412
       right       0.89      0.90      0.90       396
        stop       0.92      0.90      0.91       411
          go       0.73      0.70      0.72       402

    accuracy                           0.82      4074
   macro avg       0.82      0.82      0.82      4074
weighted avg       0.82      0.82      0.82      4074



## Model Conversion to TensorFlow Lite

This section converts the trained Keras model into TensorFlow Lite format for deployment on embedded devices.

### Steps performed:

- Convert the trained model using **TensorFlow Lite Converter**  
- Generate a **float (non-quantized) TFLite model**  
- Save the model as a `.tflite` file  

### Output

- **`speech_commands_mfcc.tflite`**  
  This file contains the trained model in TensorFlow Lite format and is used for deployment on the ESP32.

### Result

The exported TFLite model is ready to be converted into a C array (`model.h`) and integrated into the ESP32 firmware using TensorFlow Lite Micro.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)

tflite_model = converter.convert()

# Save TFLite model
with open("speech_commands_mfcc.tflite", "wb") as f:
    f.write(tflite_model)

print("FLOAT TFLite model saved: speech_commands_mfcc.tflite")

Saved artifact at '/tmp/tmpuq09o5e8'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 100, 13, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  137570662964240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137570662963088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137570662960400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137570662965008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137570662962512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137570662964624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137570662960592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137570662962896: TensorSpec(shape=(), dtype=tf.resource, name=None)
FLOAT TFLite model saved: speech_commands_mfcc.tflite
